In [0]:
import os
print("DATABRICKS_RUNTIME_VERSION:",os.environ.get('DATABRICKS_RUNTIME_VERSION',None))

In [0]:
print(spark.version)

In [0]:
print("Helo World!")

In [0]:
%sql
select "Hello World from SQL!"

# title1
## title2
### title3

text with a **bold** and *italicized* in it.

Ordered list
1. first
2. second
3. third

Unordered list
* coffee
* tea
* milk

Images:
![Associate-badge](https://www.databricks.com/wp-content/uploads/2022/04/associate-badge-eng.svg)

And of course, tables:

| user_id | user_name |
|---------|-----------|
|    1    |    Adam   |
|    2    |    Sarah  |
|    3    |    John   |

Links (or Embedded HTML): <a href="https://docs.databricks.com/notebooks/notebooks-manage.html" target="_blank"> Managing Notebooks documentation </a>

In [0]:
%run ./includes/Setup

In [0]:
print(full_name)

In [0]:
%fs ls '/databricks-datasets'

In [0]:
dbutils.help()

In [0]:
dbutils.fs.help()

In [0]:
files = dbutils.fs.ls('/databricks-datasets')
print(files)
display(files)

# Medium Draft: Predicting and Analyzing Cloud Costs with Terraform, Infracost, and PySpark

## Introduction

When deploying cloud infrastructure, the bill often arrives **after** the resources are up and running — and sometimes it’s not what you expected.  
But what if you could:
1. Estimate your costs **before** deploying.
2. Store those estimates for later analysis.
3. Compare them with actual costs months later.
4. Identify differences due to discounts, commitment plans, and shared resource allocation.

This article walks through a **Day 0 to Month 6 cost analysis workflow** using:
- **Terraform** for Infrastructure as Code (IaC)
- **Infracost** for cost estimation
- **AWS S3** (or Azure Blob) for storage
- **PySpark** for large-scale cost analysis
- **Jupyter Notebook** for interactive analytics

We’ll also cover how you can demo this project to showcase your hands-on skills to hiring managers.

---

## Architecture Overview

![Architecture](./includes/architecture.png)
*Figure 1: From Terraform Plan to PySpark Analysis.*

**Workflow steps:**
1. **Terraform Plan** generates a JSON execution plan.
2. **Infracost** processes the plan and outputs Day 0 cost estimates.
3. Estimates are stored in **S3/Blob Storage** for later retrieval.
4. After 6 months, actual costs are simulated (considering discounts and shared costs).
5. **PySpark** compares estimates with actuals to identify changes.

---

## Step-by-Step Implementation

### 1. Prepare Terraform Plan
We’ll start with a simple VPC + EC2/EKS + RDS setup.

```bash
terraform init
terraform plan -out tfplan.binary
terraform show -json tfplan.binary > plan.json
```

### 2. Run Infracost for Day 0 Estimates
```bash
infracost breakdown --path plan.json --format json --out-file infracost_day0.json
```

This gives us a detailed breakdown of estimated monthly costs.

### 3. Upload Estimates to S3 (or Azure Blob)
```bash
aws s3 cp infracost_day0.json s3://my-cost-bucket/demo/
```

### 4. Simulate Month 6 Costs
We adjust Day 0 estimates to reflect real-world changes:
- Reserved/Committed usage discounts.
- Instance family discounts.
- Kubernetes shared cost allocation.

Run the script:
```bash
python scripts/simulate_month6.py infracost_day0.json infracost_month6_actual.json
aws s3 cp infracost_month6_actual.json s3://my-cost-bucket/demo/
```

### 5. Analyze with PySpark in Jupyter Notebook
```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("InfracostAnalysis").getOrCreate()

day0_df = spark.read.json("s3a://my-cost-bucket/demo/infracost_day0.json")
month6_df = spark.read.json("s3a://my-cost-bucket/demo/infracost_month6_actual.json")

day0_flat = day0_df.selectExpr("explode(projects) as proj")     .selectExpr("explode(proj.breakdown.resources) as res")     .select(col("res.name").alias("resource"), col("res.monthlyCost").alias("day0_cost"))

month6_flat = month6_df.selectExpr("explode(projects) as proj")     .selectExpr("explode(proj.breakdown.resources) as res")     .select(col("res.name").alias("resource"), col("res.monthlyCost").alias("month6_cost"))

diff_df = day0_flat.join(month6_flat, "resource")     .withColumn("diff", col("month6_cost") - col("day0_cost"))     .withColumn("pct_change", (col("diff") / col("day0_cost")) * 100)

diff_df.show()
```

---

## Why This Project is Valuable for Hiring Managers

- **Shows hands-on IaC + FinOps skills**: Not just deploying, but optimizing for cost.
- **Demonstrates automation**: CI/CD pipelines can be extended to include cost checks.
- **End-to-end workflow**: From estimation to data analysis.
- **Scalable analysis**: PySpark can handle thousands of resources across multiple accounts.

**Extra idea:** You can integrate this into a GitHub Action to automatically fail a PR if costs exceed a certain threshold.

---

## Repository Structure

```
finops-iac-demo/
├── README.md
├── architecture.png
├── terraform/
│   └── main.tf
├── scripts/
│   └── simulate_month6.py
├── notebooks/
│   └── cost_analysis.ipynb
```

---

## Conclusion

By implementing this workflow, you can:
- Predict cloud costs before spending a dollar.
- Capture cost baselines for later analysis.
- Quantify the impact of long-term optimizations.

This **practical + analytical** approach can give you an edge in cloud engineering and FinOps-focused interviews.

You can download and run the full demo from the GitHub repository (or the zip provided).  
Give it a try, tweak the Terraform config, and see how your cost forecasts compare!

---

*Author’s note:* If you’re applying for a cloud or DevOps role, showcasing this project is a great way to prove you can think beyond deployment — straight into financial accountability.
